# Notebook 01: Exploratory Data Analysis & Temporal Windows

**FairDrift: Continuous Algorithmic Fairness Monitoring under Distribution Drift**

This notebook:
1. Loads and preprocesses the Diabetes 130-US Hospitals dataset
2. Validates demographic distributions against thesis methodology claims
3. Constructs 5 temporal windows and validates progressive drift
4. Confirms AA proportion shifts from ~21.5% (W1) to ~12.8% (W5)
5. Produces descriptive tables and figures for thesis Chapter 4


In [ ]:
# === Environment Setup ===
import sys, os

# Auto-detect Colab vs local
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    sys.path.insert(0, '/content/FairDrift-code')
    print('Running on Google Colab')
else:
    sys.path.insert(0, os.path.abspath('..'))
    print('Running locally')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ks_2samp

from src import config
from src.preprocessing import load_data, create_targets, preprocess_full
from src.temporal_windows import (
    construct_windows, validate_temporal_ordering,
    compute_subgroup_sizes, demographic_shift_report,
    subgroup_drift_comparison
)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 11
print(f'Data path: {config.DATA_PATH}')


## 1. Load Raw Data

In [ ]:
df_raw = load_data()
print(f'Dataset: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')
df_raw.head(3)


## 2. Demographic Distributions

In [ ]:
# Race distribution
print('RACE DISTRIBUTION')
print('=' * 50)
for race in ['Caucasian', 'AfricanAmerican', 'Hispanic', 'Asian', 'Other', '?']:
    n = (df_raw['race'] == race).sum()
    pct = n / len(df_raw) * 100
    print(f'  {race:<20} {n:>7,} ({pct:>5.1f}%)')

# Gender
print('\nGENDER DISTRIBUTION')
for g in df_raw['gender'].unique():
    n = (df_raw['gender'] == g).sum()
    print(f'  {g:<20} {n:>7,} ({n/len(df_raw)*100:.1f}%)')

# Age
print('\nAGE BRACKETS')
for age in sorted(df_raw['age'].unique()):
    n = (df_raw['age'] == age).sum()
    print(f'  {age:<20} {n:>7,} ({n/len(df_raw)*100:.1f}%)')


## 3. Prediction Task Base Rates

In [ ]:
df = create_targets(df_raw)

print('PREDICTION TASK BASE RATES')
print('=' * 60)
for task, info in config.TASKS.items():
    rate = df[task].mean()
    print(f'  {info["description"]:<40} {rate*100:.2f}%')

# By Race
print('\nBASE RATES BY RACE')
print(f'{"Race":<18} {"Readmit30":>10} {"ExtStay":>10} {"MedChange":>10}')
print('-' * 50)
for race in ['Caucasian', 'AfricanAmerican', 'Hispanic', 'Asian']:
    s = df[df['race'] == race]
    print(f'{race:<18} {s["readmit_30"].mean()*100:>9.2f}% '
          f'{s["extended_stay"].mean()*100:>9.2f}% '
          f'{s["med_change"].mean()*100:>9.2f}%')


## 4. Temporal Window Construction

In [ ]:
windows = construct_windows(df)
print(f'Created {len(windows)} windows of ~{len(windows[0]):,} encounters each')
for i, w in enumerate(windows):
    print(f'  W{i+1}: n={len(w):,}, encounter_id [{w["encounter_id"].min():,} - {w["encounter_id"].max():,}]')


In [ ]:
# Key validation: demographic shift
demo = demographic_shift_report(windows)
pivot = demo.pivot(index='window', columns='race', values='pct')

print('RACE % PER WINDOW')
print(pivot[['Caucasian', 'AfricanAmerican', 'Hispanic']].round(1))

print(f'\nAA shift: W1={pivot.loc[1,"AfricanAmerican"]:.1f}% -> W5={pivot.loc[5,"AfricanAmerican"]:.1f}%')


In [ ]:
# Figure: Demographic composition (publication-quality)
import matplotlib.ticker as mticker

fig, ax = plt.subplots(figsize=(10, 6.5))

races_raw = ['Caucasian', 'AfricanAmerican', 'Hispanic', 'Asian', 'Other']
race_labels = ['Caucasian', 'African American', 'Hispanic', 'Asian', 'Other']
colors = ['#3674B5', '#E8923F', '#D94F4F', '#52A6A6', '#999999']

x = np.arange(5)
bar_width = 0.55
window_labels = ['W1\n(Training)', 'W2', 'W3', 'W4', 'W5']

# Compute percentages
race_data = {}
for race in races_raw:
    race_data[race] = [pivot.loc[w+1, race] if race in pivot.columns else 0 for w in range(5)]

# Stacked bars with percentage labels
bottom = np.zeros(5)
for race, label, color in zip(races_raw, race_labels, colors):
    vals = np.array(race_data[race])
    ax.bar(x, vals, bar_width, bottom=bottom, label=label, color=color,
           edgecolor='white', linewidth=0.8)
    # Label inside bar if slice > 4%
    for j, (v, b) in enumerate(zip(vals, bottom)):
        if v > 4:
            ax.text(x[j], b + v/2, f'{v:.1f}%',
                    ha='center', va='center', fontsize=9, fontweight='bold', color='white')
    bottom += vals

# Formatting
ax.set_xticks(x)
ax.set_xticklabels(window_labels, fontsize=11)
ax.set_ylabel('Proportion (%)', fontsize=12, fontweight='bold')
ax.set_xlabel('Temporal Window', fontsize=12, fontweight='bold')
ax.set_title('Demographic Composition Shift Across Temporal Windows',
             fontsize=14, fontweight='bold', pad=15)
ax.set_ylim(0, 108)
ax.yaxis.set_major_locator(mticker.MultipleLocator(20))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.yaxis.grid(True, linestyle='--', alpha=0.3, color='gray')
ax.set_axisbelow(True)

# Legend below the chart
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=5, fontsize=10,
          frameon=True, fancybox=True, edgecolor='#cccccc', borderpad=0.8, columnspacing=1.5)

# Annotation: AA shift
ax.annotate('African American:\n21.5% (W1) → 12.8% (W5)',
            xy=(4, 81.1 + 12.8/2), xytext=(3.6, 105),
            fontsize=9, fontstyle='italic', color='#E8923F', ha='center',
            arrowprops=dict(arrowstyle='->', color='#E8923F', lw=1.5))

plt.tight_layout()
os.makedirs('../outputs/figures', exist_ok=True)
plt.savefig('../outputs/figures/fig01_demographic_composition.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

## 5. Drift Validation (KS Statistics)

In [ ]:
ks_results = validate_temporal_ordering(windows)
print('KS STATISTICS: W1 vs subsequent windows')
print(ks_results[['feature', 'ks_w1_w2', 'ks_w1_w3', 'ks_w1_w4', 'ks_w1_w5']].to_string(index=False))


In [ ]:
# Figure: KS heatmap
fig, ax = plt.subplots(figsize=(8, 5))
ks_matrix = ks_results.set_index('feature')[['ks_w1_w2', 'ks_w1_w3', 'ks_w1_w4', 'ks_w1_w5']]
ks_matrix.columns = ['W1 vs W2', 'W1 vs W3', 'W1 vs W4', 'W1 vs W5']
sns.heatmap(ks_matrix, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax, vmin=0, linewidths=0.5)
ax.set_title('Distribution Drift: KS Statistics Across Temporal Windows')
plt.tight_layout()
plt.savefig('../outputs/figures/fig02_ks_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()


## 6. Subgroup-Level Drift Comparison

In [ ]:
drift_comp = subgroup_drift_comparison(windows)
print('DRIFT COMPARISON: Caucasian vs African American (W1 vs W5)')
print(drift_comp.to_string(index=False))
print(f'\nFeatures where AA drift > 1.5x Caucasian: {(drift_comp["ratio_aa_cau"] > 1.5).sum()}')


## 7. Subgroup Sizes (Statistical Power)

In [ ]:
sizes = compute_subgroup_sizes(windows)
race_sizes = sizes[sizes['attribute'] == 'race'].pivot(index='group', columns='window', values='n')
print('Subgroup sizes per window (race):')
print(race_sizes)
insufficient = sizes[~sizes['sufficient']]
print(f'\nBelow minimum ({config.MIN_SUBGROUP_SIZE}): {len(insufficient)} combinations')
if len(insufficient) > 0:
    print(insufficient[['attribute', 'group', 'window', 'n']].to_string(index=False))


## 8. Full Preprocessing & Save Checkpoint

In [ ]:
# Full preprocessing pipeline
df_processed, feature_cols = preprocess_full()
print(f'\nTotal features for modeling: {len(feature_cols)}')

# Construct windows from processed data
windows_processed = construct_windows(df_processed)

# Save checkpoint for next notebook
import pickle
os.makedirs('../outputs', exist_ok=True)
checkpoint = {
    'df': df_processed,
    'feature_cols': feature_cols,
    'windows': windows_processed,
}
with open('../outputs/checkpoint_01.pkl', 'wb') as f:
    pickle.dump(checkpoint, f)
print('Checkpoint saved: outputs/checkpoint_01.pkl')
